# 07 - LoRA / QLoRA Fine-tune Same LLM

Fine-tunes the same exact LLM checkpoint used by Base RAG. This preserves the required fair comparison: Base RAG uses the base checkpoint, Fine-tuned RAG uses the same checkpoint plus a LoRA adapter.

In [ ]:
from pathlib import Path
import json
import sys
import importlib

try:
    from google.colab import drive
    drive.mount('/content/drive')
except ModuleNotFoundError:
    print('Not running in Google Colab; using local filesystem paths.')

DRIVE_ROOT = Path('/content/drive/MyDrive/rag')
sys.path = [str(DRIVE_ROOT)] + [p for p in sys.path if p != str(DRIVE_ROOT)]
for name in list(sys.modules):
    if name == 'src' or name.startswith('src.'):
        del sys.modules[name]
importlib.invalidate_caches()

config = json.loads((DRIVE_ROOT / 'project_config.json').read_text(encoding='utf-8'))
model_name = config.get('base_llm_model', 'google/gemma-2-2b-it')
train_jsonl = DRIVE_ROOT / 'data/processed/finetune_train_combined.jsonl'
val_jsonl = DRIVE_ROOT / 'data/processed/finetune_val_combined.jsonl'

for path in [train_jsonl, val_jsonl]:
    if not path.exists():
        raise FileNotFoundError(path)

model_name, train_jsonl, val_jsonl

In [ ]:
import importlib.util
import subprocess
import sys

required_modules = {
    'transformers': 'transformers',
    'accelerate': 'accelerate',
    'bitsandbytes': 'bitsandbytes',
    'peft': 'peft',
    'torch': 'torch',
}

missing_packages = [package for module, package in required_modules.items() if importlib.util.find_spec(module) is None]
if missing_packages:
    print('Installing missing packages:', missing_packages)
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing_packages])
else:
    print('All fine-tuning dependencies are already installed.')

If the model is gated, login to Hugging Face before training:

```python
from huggingface_hub import login
login()
```

In [ ]:
from src.finetune_lora import train_lora

# Smoke train: verifies memory, tokenization, LoRA wiring, and adapter saving.
smoke_config = train_lora(
    train_jsonl=train_jsonl,
    val_jsonl=val_jsonl,
    output_dir=DRIVE_ROOT / 'models/adapters/gemma_2_2b_it_lora_smoke',
    model_name=model_name,
    max_length=1536,
    max_train_samples=200,
    max_val_samples=50,
    num_train_epochs=1.0,
    learning_rate=2e-4,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    logging_steps=10,
    eval_steps=25,
    save_steps=50,
    lora_r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    use_4bit=True,
)

smoke_config

Run this medium training cell after the smoke train finishes successfully. This is the recommended next run for Colab: it trains on 4,000 examples and saves the adapter that should be used for the first full Fine-tuned RAG evaluation.

In [ ]:
medium_config = train_lora(
    train_jsonl=train_jsonl,
    val_jsonl=val_jsonl,
    output_dir=DRIVE_ROOT / 'models/adapters/gemma_2_2b_it_lora_combined_v1',
    model_name=model_name,
    max_length=1536,
    max_train_samples=4000,
    max_val_samples=400,
    num_train_epochs=1.0,
    learning_rate=2e-4,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    logging_steps=25,
    eval_steps=200,
    save_steps=400,
    lora_r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    use_4bit=True,
)

medium_config

Optional final full-data run. Use this only if the medium adapter improves Fine-tuned RAG and Colab time is enough. This writes a separate adapter so the medium result is preserved.

In [ ]:
# Optional full-data run. Uncomment only after medium training/evaluation is successful.
# final_config = train_lora(
#     train_jsonl=train_jsonl,
#     val_jsonl=val_jsonl,
#     output_dir=DRIVE_ROOT / 'models/adapters/gemma_2_2b_it_lora_combined_full_v1',
#     model_name=model_name,
#     max_length=1536,
#     max_train_samples=None,
#     max_val_samples=None,
#     num_train_epochs=1.0,
#     learning_rate=2e-4,
#     per_device_train_batch_size=1,
#     per_device_eval_batch_size=1,
#     gradient_accumulation_steps=8,
#     logging_steps=50,
#     eval_steps=500,
#     save_steps=1000,
#     lora_r=16,
#     lora_alpha=32,
#     lora_dropout=0.05,
#     use_4bit=True,
# )
# final_config

Expected adapter outputs:

- `models/adapters/gemma_2_2b_it_lora_smoke/`
- `models/adapters/gemma_2_2b_it_lora_combined_v1/` for the full run

The fine-tuned RAG evaluation must use the same retrieval corpus, retriever mode, top-k context, prompt, and decoding settings as Base RAG. Only the LoRA adapter changes.